In [1]:
import pandas as pd

In [2]:
nav = pd.read_csv("data/raw/02_nav_history.csv")
nav.head()

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [3]:
nav.dtypes

amfi_code      int64
date          object
nav          float64
dtype: object

In [4]:
nav["date"] = pd.to_datetime(nav["date"])

In [5]:
nav.dtypes

amfi_code             int64
date         datetime64[ns]
nav                 float64
dtype: object

In [6]:
nav = nav.sort_values(["amfi_code", "date"]).reset_index(drop=True)

In [7]:
len(nav)

46000

In [8]:
nav = nav.drop_duplicates(subset=["amfi_code", "date"], keep="last")

In [9]:
print(f"Removed {46000 - len(nav)} duplicate (amfi_code, date) rows")

Removed 0 duplicate (amfi_code, date) rows


In [10]:
invalid = nav[nav["nav"] <= 0]
invalid

,amfi_code,date,nav


In [11]:
print(f"Found {len(invalid)} rows with NAV <= 0")

Found 0 rows with NAV <= 0


In [12]:
nav = nav[nav["nav"] > 0]
nav.head()

,amfi_code,date,nav
0,100016,2022-01-03,520.4608
1,100016,2022-01-04,515.0971
2,100016,2022-01-05,521.7239
3,100016,2022-01-06,515.7880
4,100016,2022-01-07,515.1639


In [13]:
filled_frames = []
for code, group in nav.groupby("amfi_code"):
    full_range = pd.date_range(group["date"].min(), group["date"].max(), freq="D")
    group = group.set_index("date").reindex(full_range)
    group["amfi_code"] = code
    group["nav"] = group["nav"].ffill()
    filled_frames.append(group)

In [14]:
filled_frames

[            amfi_code       nav
 2022-01-03     100016  520.4608
 2022-01-04     100016  515.0971
 2022-01-05     100016  521.7239
 2022-01-06     100016  515.7880
 2022-01-07     100016  515.1639
 ...               ...       ...
 2026-05-25     100016  599.9335
 2026-05-26     100016  596.4426
 2026-05-27     100016  590.8555
 2026-05-28     100016  590.9452
 2026-05-29     100016  583.6113
 
 [1608 rows x 2 columns],
             amfi_code      nav
 2022-01-03     100025  26.3169
 2022-01-04     100025  26.2234
 2022-01-05     100025  26.2221
 2022-01-06     100025  26.1728
 2022-01-07     100025  26.2261
 ...               ...      ...
 2026-05-25     100025  31.9938
 2026-05-26     100025  31.9646
 2026-05-27     100025  31.9203
 2026-05-28     100025  31.9101
 2026-05-29     100025  31.8843
 
 [1608 rows x 2 columns],
             amfi_code       nav
 2022-01-03     100033  107.3758
 2022-01-04     100033  105.9447
 2022-01-05     100033  105.4800
 2022-01-06     100033  104.9350

In [15]:
nav_clean = pd.concat(filled_frames).rename_axis("date").reset_index()

In [16]:
nav_clean = nav_clean[["amfi_code", "date", "nav"]]

In [17]:
nav_clean.head()

,amfi_code,date,nav
0,100016,2022-01-03,520.4608
1,100016,2022-01-04,515.0971
2,100016,2022-01-05,521.7239
3,100016,2022-01-06,515.7880
4,100016,2022-01-07,515.1639


In [18]:
nav_clean.to_csv("data/processed/nav_history_clean.csv", index=False)

In [19]:
txn = pd.read_csv("data/raw/08_investor_transactions.csv")
txn.head()

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [20]:
txn.dtypes

investor_id            object
transaction_date       object
amfi_code               int64
transaction_type       object
amount_inr              int64
state                  object
city                   object
city_tier              object
age_group              object
gender                 object
annual_income_lakh    float64
payment_mode           object
kyc_status             object
dtype: object

In [21]:
valid_types = {"SIP", "Lumpsum", "Redemption"}

In [22]:
txn["transaction_type"] = txn["transaction_type"].str.strip().str.title()

In [23]:
txn.loc[txn["transaction_type"].str.upper() == "SIP", "transaction_type"] = "SIP"

In [24]:
bad_types = txn[~txn["transaction_type"].isin(valid_types)]

In [25]:
print(f"Found {len(bad_types)} rows with an unrecognised transaction_type")

Found 0 rows with an unrecognised transaction_type


In [26]:
invalid_amt = txn[txn["amount_inr"] <= 0]

In [27]:
print(f"Found {len(invalid_amt)} rows with amount_inr <= 0")

Found 0 rows with amount_inr <= 0


In [28]:
txn = txn[txn["amount_inr"] > 0]

In [29]:
txn["transaction_date"] = pd.to_datetime(txn["transaction_date"], errors="coerce")

In [30]:
bad_dates = txn["transaction_date"].isna().sum()

In [31]:
print(f"Found {bad_dates} unparseable dates")

Found 0 unparseable dates


In [32]:
txn = txn.dropna(subset=["transaction_date"])

In [33]:
valid_kyc = {"Verified", "Pending"}

In [34]:
bad_kyc = txn[~txn["kyc_status"].isin(valid_kyc)]

In [35]:
print(f"Found {len(bad_kyc)} rows with unrecognised kyc_status")

Found 0 rows with unrecognised kyc_status


In [36]:
txn_clean = txn.drop_duplicates().reset_index(drop=True)

In [37]:
txn_clean.to_csv("data/processed/investor_transactions_clean.csv", index=False)

In [38]:
perf = pd.read_csv("data/raw/07_scheme_performance.csv")
perf.head()

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


In [39]:
perf.dtypes

amfi_code               int64
scheme_name            object
fund_house             object
category               object
plan                   object
return_1yr_pct        float64
return_3yr_pct        float64
return_5yr_pct        float64
benchmark_3yr_pct     float64
alpha                 float64
beta                  float64
sharpe_ratio          float64
sortino_ratio         float64
std_dev_ann_pct       float64
max_drawdown_pct      float64
aum_crore               int64
expense_ratio_pct     float64
morningstar_rating      int64
risk_grade             object
dtype: object

In [40]:
perf.isnull().sum()

amfi_code             0
scheme_name           0
fund_house            0
category              0
plan                  0
return_1yr_pct        0
return_3yr_pct        0
return_5yr_pct        0
benchmark_3yr_pct     0
alpha                 0
beta                  0
sharpe_ratio          0
sortino_ratio         0
std_dev_ann_pct       0
max_drawdown_pct      0
aum_crore             0
expense_ratio_pct     0
morningstar_rating    0
risk_grade            0
dtype: int64

In [41]:
numeric_cols = [
    "return_1yr_pct", "return_3yr_pct", "return_5yr_pct", "benchmark_3yr_pct",
    "alpha", "beta", "sharpe_ratio", "sortino_ratio",
    "std_dev_ann_pct", "max_drawdown_pct", "expense_ratio_pct",
]

for col in numeric_cols:
    perf[col] = pd.to_numeric(perf[col], errors="coerce")
    n_bad = perf[col].isna().sum()
    if n_bad:
        print(f"{col}: {n_bad} non-numeric values coerced to NaN")

In [42]:
perf["flag_bad_drawdown"] = perf["max_drawdown_pct"] > 0

In [43]:
perf["flag_bad_stddev"] = perf["std_dev_ann_pct"] < 0

In [44]:
anomalies = perf[perf["flag_bad_drawdown"] | perf["flag_bad_stddev"]]

In [45]:
print(f"Found {len(anomalies)} anomalous rows (see flag_* columns)")

Found 0 anomalous rows (see flag_* columns)


In [46]:
perf["flag_expense_out_of_range"] = ~perf["expense_ratio_pct"].between(0.1, 2.5)

In [47]:
print(f"Found {perf['flag_expense_out_of_range'].sum()} rows outside 0.1-2.5% expense ratio")

Found 0 rows outside 0.1-2.5% expense ratio


In [48]:
perf_clean = perf.drop_duplicates().reset_index(drop=True)

In [49]:
perf_clean.to_csv("data/processed/scheme_performance_clean.csv", index=False)